In [6]:
import numpy as np
from collections import defaultdict

# ------------------------------------------------------------
# 1. Datos de ejemplo: (vector de meta-features, regla lógica)
# Cada vector contiene: [num_muestras, num_features, num_clases, desbalanceo]
# Desbalanceo = proporción de la clase mayoritaria (0.5 = balanceado, 0.9 = muy desbalanceado)
# ------------------------------------------------------------
database = [
    {
        "vector": np.array([5000, 20, 2, 0.52]),  # dataset A
        "regla": "IF num_clases <= 2 AND desbalanceo < 0.6 THEN recomendado = 'LogisticRegression'"
    },
    {
        "vector": np.array([200, 100, 10, 0.55]),  # dataset B
        "regla": "IF num_features > 50 AND num_muestras < 500 THEN recomendado = 'RandomForest'"
    },
    {
        "vector": np.array([10000, 5, 3, 0.85]),  # dataset C (muy desbalanceado)
        "regla": "IF desbalanceo > 0.7 THEN preproceso = 'SMOTE' AND recomendado = 'XGBoost'"
    },
    {
        "vector": np.array([800, 30, 8, 0.62]),  # dataset D
        "regla": "IF num_muestras < 1000 AND num_clases > 5 THEN recomendado = 'RandomForest'"
    },
    {
        "vector": np.array([3000, 15, 4, 0.50]),  # dataset E
        "regla": "IF num_clases BETWEEN 3 AND 6 AND desbalanceo < 0.6 THEN recomendado = 'SVM'"
    }
]

# ------------------------------------------------------------
# 2. Funciones de utilidad: distancia euclidiana y k vecinos
# ------------------------------------------------------------
def distancia(v1, v2):
    return np.linalg.norm(v1 - v2)

def k_vecinos_cercanos(nuevo_vector, k=2):
    """Retorna los k índices de la base con vectores más cercanos al nuevo_vector"""
    distancias = [(i, distancia(nuevo_vector, entry["vector"])) for i, entry in enumerate(database)]
    distancias.sort(key=lambda x: x[1])
    return [idx for idx, _ in distancias[:k]]

# ------------------------------------------------------------
# 3. Motor lógico simbólico (evaluador de reglas)
# ------------------------------------------------------------

import re

def evaluar_regla(regla_str, meta_features):
    if "THEN" not in regla_str:
        return None
    cond_part, acc_part = regla_str.split("THEN")
    cond_part = cond_part.replace("IF", "").strip()

    # 1. Procesar BETWEEN: convertir a comparación encadenada con variable simbólica
    def between_replace(match):
        var, low, high = match.groups()
        low_num = float(low)
        high_num = float(high)
        # Devolvemos expresión con el nombre de la variable aún sin reemplazar
        return f"({low_num} <= {var} <= {high_num})"

    cond_part = re.sub(r'(\w+)\s+BETWEEN\s+([0-9.]+)\s+AND\s+([0-9.]+)', between_replace, cond_part)

    # 2. Reemplazar variables por sus valores numéricos
    for var, val in meta_features.items():
        num_val = float(val)  # convertir a float estándar
        # Reemplazar palabra completa (borde de palabra)
        cond_part = re.sub(r'\b' + var + r'\b', str(num_val), cond_part)

    # 3. Convertir operadores lógicos a sintaxis Python
    cond_part = cond_part.replace(' AND ', ' and ')
    cond_part = cond_part.replace(' OR ', ' or ')
    # También manejar casos sin espacios (p.ej., "x>1 AND y<2")
    cond_part = cond_part.replace('AND', 'and').replace('OR', 'or')

    # 4. Evaluar la condición (segura porque solo contiene números y operadores)
    try:
        cond_cumple = eval(cond_part)
    except Exception as e:
        print(f"Error evaluando condición: {cond_part}\n{e}")
        cond_cumple = False

    if not cond_cumple:
        return None

    # 5. Parsear acciones (parte THEN)
    acciones = {}
    for parte in acc_part.split("AND"):
        parte = parte.strip()
        if "=" in parte:
            var, valor = parte.split("=", 1)
            var = var.strip()
            valor = valor.strip().strip("'")
            acciones[var] = valor
    return acciones

# ------------------------------------------------------------
# 4. Unificación de reglas (combinar conclusiones de múltiples reglas)
# ------------------------------------------------------------
def unificar_reglas(lista_acciones):
    """
    Recibe una lista de diccionarios (cada uno con conclusiones de una regla)
    y combina las recomendaciones. Ejemplo:
        [{'recomendado': 'RF'}, {'recomendado': 'SVM', 'preproceso': 'SMOTE'}]
    -> Si hay conflictos, se usa la mayoría simple.
    """
    unificado = {}
    for acc in lista_acciones:
        for clave, valor in acc.items():
            if clave not in unificado:
                unificado[clave] = []
            unificado[clave].append(valor)
    # Votación por mayoría (o primera si empate)
    resultado = {}
    for clave, valores in unificado.items():
        from collections import Counter
        conteo = Counter(valores)
        mas_comun = conteo.most_common(1)[0][0]
        resultado[clave] = mas_comun
    return resultado

# ------------------------------------------------------------
# 5. Ejemplo completo
# ------------------------------------------------------------
def recomendar_pipeline(nuevo_vector, k=2):
    # Paso 1: encontrar k vectores más cercanos
    indices = k_vecinos_cercanos(nuevo_vector, k)
    print(f"Índices de los {k} datasets más similares: {indices}")
    
    # Paso 2: recuperar las reglas asociadas
    reglas_recuperadas = [database[i]["regla"] for i in indices]
    print("\nReglas recuperadas:")
    for r in reglas_recuperadas:
        print(f"  - {r}")
    
    # Paso 3: aplicar motor lógico sobre el nuevo dataset (usando sus meta-features)
    # Asumimos que el nuevo vector tiene el mismo orden: [num_muestras, num_features, num_clases, desbalanceo]
    meta_new = {
        "num_muestras": nuevo_vector[0],
        "num_features": nuevo_vector[1],
        "num_clases": nuevo_vector[2],
        "desbalanceo": nuevo_vector[3]
    }
    print(f"\nMeta-features del nuevo dataset: {meta_new}")
    
    acciones_que_cumplen = []
    for regla in reglas_recuperadas:
        resultado = evaluar_regla(regla, meta_new)
        if resultado:
            acciones_que_cumplen.append(resultado)
            print(f"  ✓ Regla cumplida: {regla} -> conclusiones: {resultado}")
        else:
            print(f"  ✗ Regla no cumplida: {regla}")
    
    # Paso 4: unificar las conclusiones
    if not acciones_que_cumplen:
        print("\nNo se cumplió ninguna regla. No hay recomendación.")
        return None
    else:
        recomendacion_final = unificar_reglas(acciones_que_cumplen)
        print(f"\n🔮 Recomendación unificada: {recomendacion_final}")
        return recomendacion_final

# ------------------------------------------------------------
# Prueba con un nuevo dataset
# ------------------------------------------------------------
if __name__ == "__main__":
    # Nuevo dataset: 2500 muestras, 25 features, 5 clases, desbalanceo 0.58
    nuevo = np.array([2500, 25, 5, 0.58])
    print("=== NUEVO DATASET ===")
    print(f"Vector: {nuevo}\n")
    
    resultado = recomendar_pipeline(nuevo, k=2)
    
    # (Opcional) Aquí podrías llamar a un LLM para generar el pipeline en código
    if resultado:
        print("\n--- Generación de pipeline por LLM (mock) ---")
        print(f"Según la recomendación '{resultado.get('recomendado', 'ninguno')}', un pipeline típico sería:")
        print(f"  from sklearn.{resultado.get('recomendado', 'dummy')} import ...")
        # En un caso real, aquí invocarías a OpenAI o a un modelo local

=== NUEVO DATASET ===
Vector: [2.5e+03 2.5e+01 5.0e+00 5.8e-01]

Índices de los 2 datasets más similares: [4, 3]

Reglas recuperadas:
  - IF num_clases BETWEEN 3 AND 6 AND desbalanceo < 0.6 THEN recomendado = 'SVM'
  - IF num_muestras < 1000 AND num_clases > 5 THEN recomendado = 'RandomForest'

Meta-features del nuevo dataset: {'num_muestras': np.float64(2500.0), 'num_features': np.float64(25.0), 'num_clases': np.float64(5.0), 'desbalanceo': np.float64(0.58)}
  ✓ Regla cumplida: IF num_clases BETWEEN 3 AND 6 AND desbalanceo < 0.6 THEN recomendado = 'SVM' -> conclusiones: {'recomendado': 'SVM'}
  ✗ Regla no cumplida: IF num_muestras < 1000 AND num_clases > 5 THEN recomendado = 'RandomForest'

🔮 Recomendación unificada: {'recomendado': 'SVM'}

--- Generación de pipeline por LLM (mock) ---
Según la recomendación 'SVM', un pipeline típico sería:
  from sklearn.SVM import ...
